In [0]:
%sql

CREATE CATALOG IF NOT EXISTS atliq MANAGED LOCATION 'abfss://lakehouse@atliqcommercewarehouse.dfs.core.windows.net/';
CREATE SCHEMA  IF NOT EXISTS atliq.silver;
CREATE SCHEMA  IF NOT EXISTS atliq.gold;


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType, DateType, TimestampType

BRONZE = "abfss://lakehouse@atliqcommercewarehouse.dfs.core.windows.net/bronze"

customers = (spark.read.parquet(f"{BRONZE}/customers")
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("signup_date", F.to_date("signup_date"))
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull()))

customers.write.format("delta").mode("overwrite").saveAsTable("atliq.silver.customers")


In [0]:
products = (
    spark.read.parquet(f"{BRONZE}/products")
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("unit_price", F.col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .dropDuplicates(["product_id"])
    .filter(
        F.col("product_id").isNotNull() & 
        (F.col("unit_price") >= 0)
    )
)

products.write.format("delta").mode("overwrite").saveAsTable("atliq.silver.products")

In [0]:
supplier_price_list = (
    spark.read.parquet(f"{BRONZE}/supplier_price_list")
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("supplier_name", F.trim(F.col("supplier_name")))
    .withColumn("supplier_cost", F.col("supplier_cost").cast(DecimalType(10, 2)))
    .withColumn("effective_date", F.to_date("effective_date"))
    .filter(F.col("product_id").isNotNull())
)

supplier_price_list.write.format("delta").mode("overwrite").saveAsTable("atliq.silver.supplier_price_list")

In [0]:
marketing_spend = (
    spark.read.parquet(f"{BRONZE}/marketing_spend")
    .withColumn("spend_date", F.to_date("spend_date"))
    .withColumn("channel", F.trim(F.col("channel")))
    .withColumn("campaign", F.trim(F.col("campaign")))
    .withColumn("spend_amount", F.col("spend_amount").cast(DecimalType(12, 2)))
    .withColumn("clicks", F.col("clicks").cast(IntegerType()))
    .filter(F.col("spend_date").isNotNull())
)

marketing_spend.write.format("delta").mode("overwrite").saveAsTable("atliq.silver.marketing_spend")

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

run_date = dbutils.widgets.get("run_date")           # passed in by ADF/Workflows
src_path = f"{BRONZE}/orders/ingest_date={run_date}"

batch = spark.read.parquet(src_path)

# keep the latest version of each order in this batch
w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
src = (batch.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
            .withColumn("order_date", F.to_date("order_date"))
            .withColumn("order_amount", F.col("order_amount").cast("decimal(12,2)"))
            .filter(F.col("order_id").isNotNull()))

# make sure the target exists (first run), then upsert on the business key
(DeltaTable.createIfNotExists(spark)
    .tableName("atliq.silver.orders").addColumns(src.schema).execute())

(DeltaTable.forName(spark, "atliq.silver.orders").alias("t")
    .merge(src.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
    .whenNotMatchedInsertAll()
    .execute())


In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

run_date = dbutils.widgets.get("run_date")  # passed in by ADF/Workflows
src_path = f"{BRONZE}/order_items/ingest_date={run_date}"

batch = spark.read.parquet(src_path)

# Keep the latest version of each line item in this batch using created_at
w = Window.partitionBy("order_item_id").orderBy(F.col("created_at").desc())
src = (
    batch.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("item_price", F.col("item_price").cast("decimal(10,2)"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .filter(
        F.col("order_item_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("product_id").isNotNull()
        & (F.col("quantity") > 0)
    )
)

# Ensure target table exists (first run)
(
    DeltaTable.createIfNotExists(spark)
    .tableName("atliq.silver.order_items")
    .addColumns(src.schema)
    .execute()
)

# Upsert on the business key (order_item_id)
(
    DeltaTable.forName(spark, "atliq.silver.order_items")
    .alias("t")
    .merge(src.alias("s"), "t.order_item_id = s.order_item_id")
    .whenMatchedUpdateAll(condition="s.created_at > t.created_at")
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

run_date = dbutils.widgets.get("run_date")  # passed in by ADF/Workflows
src_path = f"{BRONZE}/payments/ingest_date={run_date}"

batch = spark.read.parquet(src_path)

# Keep the latest version of each payment in this batch
w = Window.partitionBy("payment_id").orderBy(F.col("updated_at").desc())
src = (
    batch.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("payment_id", F.col("payment_id").cast("int"))
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("method", F.upper(F.trim(F.col("method"))))
    .withColumn("paid_at", F.to_timestamp("paid_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .filter(
        F.col("payment_id").isNotNull()
        & F.col("order_id").isNotNull()
        & (F.col("amount") >= 0)
    )
)

# Ensure target table exists (first run)
(
    DeltaTable.createIfNotExists(spark)
    .tableName("atliq.silver.payments")
    .addColumns(src.schema)
    .execute()
)

# Upsert on the business key (payment_id)
(
    DeltaTable.forName(spark, "atliq.silver.payments")
    .alias("t")
    .merge(src.alias("s"), "t.payment_id = s.payment_id")
    .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
    .whenNotMatchedInsertAll()
    .execute()
)